# Current Rainier comparisons
Run All. Uses only the 13 catalog inputs in this folder; no historical archives or external paths are required. Matching is one-to-one at ±7 s, without a clock correction. The possible ELEP 88/89 duplicate remains unconsolidated. See README.md for population definitions and limitations.

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import csv,json,math,os,tempfile
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching
from IPython.display import display,Image
os.environ.setdefault('MPLCONFIGDIR',str(Path(tempfile.gettempdir())/'rainier_professional_mpl'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
BASE=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'CURRENT_FILES.json').is_file()),None)
if BASE is None:raise FileNotFoundError('Run inside phasepickerresults.')
FILES=json.loads((BASE/'CURRENT_FILES.json').read_text())
TOLERANCE_SECONDS=7.0
START='2023-08-23T21:00:00Z';END='2023-11-22T00:00:00Z'
OUT=BASE/'generated';OUT.mkdir(exist_ok=True)
def sec(value):
 t=datetime.fromisoformat(value.replace('Z','+00:00'))
 if t.tzinfo is None:t=t.replace(tzinfo=timezone.utc)
 return t.timestamp()
def records(role,time_key,id_key,confirmed_only=False):
 with (BASE/FILES[role]['path']).open(newline='') as f:rows=list(csv.DictReader(f))
 return [dict(r,id=str(r.get(id_key,i)),seconds=sec(r[time_key])) for i,r in enumerate(rows)
         if sec(START)<=sec(r[time_key])<sec(END) and (not confirmed_only or r['confirm']=='yes')]
P=records('pyocto_reviewed','time','idx')
E=records('elep_review','time_utc','idx',True)
C=records('comcat_regional','time_utc','id')
G=records('gamma_reviewed','time','event_index')
G5=records('gamma_five','time','event_index')
GA=records('gamma_all','time','event_index')
TD=records('tm_das_selected','UTC','row_id')
TS=records('tm_station_raw','event_time_utc','row_id')
print({'PhaseNet_PyOcto':len(P),'ELEP_PyOcto_accepted':len(E),'ComCat_regional':len(C),
       'PhaseNet_GaMMA':len(G),'five_station_GaMMA':len(G5),'all_station_GaMMA':len(GA),
       'DAS_TM':len(TD),'station_TM':len(TS)})

In [ ]:
def match(left,right):
 delta=np.array([r['seconds'] for r in right])[None,:]-np.array([r['seconds'] for r in left])[:,None]
 eligible=np.abs(delta)<=TOLERANCE_SECONDS
 ii,jj=linear_sum_assignment(np.where(eligible,np.abs(delta)/(TOLERANCE_SECONDS+1),1e6))
 pairs=[(int(i),int(j)) for i,j in zip(ii,jj) if eligible[i,j]]
 assert len(pairs)==int((maximum_bipartite_matching(csr_matrix(eligible),perm_type='column')>=0).sum())
 return [{'left_id':left[i]['id'],'right_id':right[j]['id'],'right_minus_left_seconds':float(delta[i,j])} for i,j in pairs]
EP=match(E,P);EC=match(E,C);PC=match(P,C)
ep={r['left_id']:r['right_id'] for r in EP};ec={r['left_id']:r['right_id'] for r in EC};pc={r['left_id']:r['right_id'] for r in PC}
both=set(ep)&set(ec);triangles={i for i in both if pc.get(ep[i])==ec[i]}
partition=[len(both),len(set(ep)-set(ec)),len(E)-len(set(ep)|set(ec))]
s={'association':'PyOcto for both','PhaseNet_DAS_reviewed_records':len(P),'ELEP_DAS_confirmed_records':len(E),
   'ComCat_reference_earthquakes':len(C),'PhaseNet_ELEP_shared':len(EP),'PhaseNet_ComCat_shared':len(PC),
   'ELEP_ComCat_shared':len(EC),'shared_by_all_three':len(triangles),
   'ELEP_PhaseNet_without_ComCat':partition[1],'ELEP_without_either_counterpart':partition[2],
   'clock_correction_applied':False,'time_tolerance_seconds':TOLERANCE_SECONDS}
print(json.dumps(s,indent=2))
extra={'Gamma_DAS_vs_five_stations':match(G,G5),'Gamma_DAS_vs_TM':match(G,TD),
       'Gamma_five_stations_vs_existing_TM':match(G5,TS)}
print('Other matches:',{name:len(pairs) for name,pairs in extra.items()})
print('ELEP nonmatch IDs:',[r['id'] for r in E if r['id'] not in set(ep)])
for name,pairs in dict(ELEP_PhaseNet=EP,ELEP_ComCat=EC,PhaseNet_ComCat=PC,**extra).items():
 with (OUT/f'{name}_pairs.csv').open('w',newline='') as f:
  w=csv.DictWriter(f,fieldnames=['left_id','right_id','right_minus_left_seconds']);w.writeheader();w.writerows(pairs)
(OUT/'current_comparison_summary.json').write_text(json.dumps(s,indent=2)+'\n')

## Current picker-overlap figure

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'text.color':'#27343E',
    'axes.labelcolor':'#27343E','xtick.color':'#45525C','ytick.color':'#27343E',
    'axes.edgecolor':'#83919B','axes.linewidth':.65,'axes.spines.top':False,
    'axes.spines.right':False,'axes.spines.left':False,'pdf.fonttype':42,'svg.fonttype':'none',
    'figure.facecolor':'white','savefig.facecolor':'white'})
BLUE='#2879A6';PURPLE='#8062A0';TEAL='#1B786D';ORANGE='#D99A37';GREY='#81909C'
fig=plt.figure(figsize=(10.8,3.65))
axes=[fig.add_axes(b) for b in [[.14,.22,.18,.67],[.47,.22,.20,.67],[.835,.22,.15,.67]]]

def bars(ax,labels,values,colors,xmax,ticks,xlabel):
    for i,(n,color) in enumerate(zip(values,colors)):
        ax.barh(i,n,height=.50,color=color,linewidth=0)
        ax.text(n+xmax*.025,i,f'{n:,}',va='center',fontsize=9)
    ax.set(yticks=range(len(labels)),yticklabels=labels,ylim=(len(labels)-.55,-.55),
           xlim=(0,xmax),xticks=ticks,xlabel=xlabel)
    ax.tick_params(axis='y',length=0,pad=7,labelsize=8.5)
    ax.tick_params(axis='x',labelsize=8,width=.65,length=3)
    ax.set_axisbelow(True);ax.grid(axis='x',color='#E6EAED',linewidth=.45)

bars(axes[0],['PhaseNet-DAS','ELEP-DAS','ComCat'],
     [s['PhaseNet_DAS_reviewed_records'],s['ELEP_DAS_confirmed_records'],s['ComCat_reference_earthquakes']],
     [BLUE,PURPLE,GREY],1120,[0,500,1000],'Catalog records')
bars(axes[1],['PhaseNet–ELEP','PhaseNet–ComCat','ELEP–ComCat','All three'],
     [s['PhaseNet_ELEP_shared'],s['PhaseNet_ComCat_shared'],s['ELEP_ComCat_shared'],s['shared_by_all_three']],
     [PURPLE,BLUE,TEAL,TEAL],255,[0,100,200],'Temporal matches')
bars(axes[2],['Both catalogs','PhaseNet only','Neither'],partition,
     [TEAL,BLUE,ORANGE],34,[0,10,20,30],'ELEP-DAS records')
for x,letter in zip([.025,.365,.73],['a','b','c']):
    fig.text(x,.945,letter,fontweight='bold',fontsize=12)
for ext in ['png','pdf','svg']:
    fig.savefig(OUT/f'PhaseNet_ELEP_ComCat_overlap_{TOLERANCE_SECONDS:g}s.{ext}',dpi=450,bbox_inches='tight',pad_inches=.06)
plt.close(fig)
display(Image(filename=str(OUT/f'PhaseNet_ELEP_ComCat_overlap_{TOLERANCE_SECONDS:g}s.png')))